In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

# 导入核心工具
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression

# 1. 加载你那份 1000 行的脏数据
df = pd.read_csv('data/dirty_music_data.csv')

# 2. 【手术第一步】清理目标变量的坑
# 只要 popularity 缺了，这行就不能用来训练
df_clean = df.dropna(subset=['popularity'])

# 3. 提取特征和标签 (把没用的 song_name 扔掉)
X = df_clean.drop(['popularity', 'song_name'], axis=1)
y = df_clean['popularity']

# 4. 【分水岭】先拆分，严防数据泄露
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 5. 【分拣中心】定义不同列的填充和转换逻辑
num_cols = ['duration_ms', 'loudness'] # 数字列：需要填坑（中位数）
cat_cols = ['genre']                  # 类别列：需要填坑（众数）+ 独热编码

# 数值预处理流水线
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median'))
])

# 类别预处理流水线
categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore')) # 忽略没见过的流派
])

# 组合成 ColumnTransformer
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, num_cols),
        ('cat', categorical_transformer, cat_cols)
    ])

# 6. 【大总管】组装最终的 Pipeline
final_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),   # 第一步：全自动填坑+编码
    ('regressor', LinearRegression()) # 第二步：线性回归模型
])

# 7. 一键运行：训练模型
final_pipeline.fit(X_train, y_train)

# 8. 查看成果
print("--- 自动化流水线运行报告 ---")
print(f"训练集得分: {final_pipeline.score(X_train, y_train):.4f}")
print(f"测试集得分: {final_pipeline.score(X_test, y_test):.4f}")

--- 自动化流水线运行报告 ---
训练集得分: 0.0173
测试集得分: -0.0136
